# Donut Fine-Tuning for SurveyPlan AI
This notebook demonstrates how to fine-tune the `naver-clova-ix/donut-base` model on a custom dataset of survey plans, parsing text-based classes (lot geometry, parcel info, tabular data, general text) from crops.


## 1. Environment & Setup
Install necessary libraries and mount Google Drive. Connecting to Google Drive allows us to load the images and save the model persistently.


In [ ]:
!pip install -q transformers datasets pytorch-lightning sentencepiece torchvision accelerate

from google.colab import drive
drive.mount('/content/drive')


## Phase I: Data Engineering Logic
First, we need to generate `metadata.jsonl` from our ground truth. We'll clean survey-specific characters (`°`, `'`, `"`) and convert them to `-` and `.` respectively to minimize Character Error Rate (CER).


In [ ]:
import os
import json
from pathlib import Path

# Paths
DATA_DIR = Path('/content/drive/MyDrive/SurveyPlan_AI/Crop_Dataset')
IMAGES_DIR = DATA_DIR / 'images'
METADATA_PATH = DATA_DIR / 'metadata.jsonl'
os.makedirs(IMAGES_DIR, exist_ok=True)

# Normalization Utility
def normalize_survey_text(text: str) -> str:
    """
    Cleans survey-specific characters.
    Example: 45° 30' 15" -> 45- 30. 15.
    """
    text = text.replace('°', '-')
    text = text.replace("'", '.')
    text = text.replace('"', '.')
    return text

def generate_metadata(annotations, output_path):
    """
    annotations: list of dicts with 'file_name' and 'gt_parse'
    where 'gt_parse' is a dictionary structured by the donut schema.
    """
    with open(output_path, 'w', encoding='utf-8') as f:
        for ann in annotations:
            # Normalize strings in gt_parse recursively or directly
            # For simplicity, assuming ann['gt_parse'] is a JSON string or dict
            if isinstance(ann['gt_parse'], dict):
                # dump to json string and replace
                gt_str = json.dumps(ann['gt_parse'], ensure_ascii=False)
                gt_str = normalize_survey_text(gt_str)
                ann['gt_parse'] = json.loads(gt_str)
                
            # Donut formatting requires ground_truth field
            record = {
                "file_name": ann['file_name'],
                "ground_truth": json.dumps({"gt_parse": ann['gt_parse']}, ensure_ascii=False)
            }
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

print("Metadata generator defined! Set your annotations and call generate_metadata() to create metadata.jsonl")


Load the dataset using Hugging Face's `datasets` library.


In [ ]:
from datasets import load_dataset

# Assuming the directory structure is:
# DATA_DIR/
# ├── metadata.jsonl
# └── image_1.jpg
# └── image_2.jpg

dataset = load_dataset("imagefolder", data_dir=str(DATA_DIR), split="train")

# You might want to split into train and test
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset['train']
val_dataset = dataset['test']

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")


## Phase II: Training Pipeline
Configure Processor & Model. We add our specific Alphabet tokens as Task Start Tokens and Schema Tokens.

Task Start Tokens: `<s_lot_geometry>`, `<s_parcel_info>`, `<s_tabular_data>`, `<s_general>`.



In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel, VisionEncoderDecoderConfig
import torch

model_id = "naver-clova-ix/donut-base"

# 1. Processor Setup with custom crop size for high fidelity text
processor = DonutProcessor.from_pretrained(model_id)
processor.image_processor.size = {"height": 1280, "width": 960}

# 2. Add custom alphabet / schema tokens
task_start_tokens = ["<s_lot_geometry>", "<s_parcel_info>", "<s_tabular_data>", "<s_general>"]
schema_tokens = [
    "<az>", "<dist>", "<curve_data>", 
    "<lot_id>", "<adj_id>", "<area_val>",
    "<row>", "<pt_id>", "<north>", "<east>", "<radius>", "<arc>", "<delta>", "<chord>",
    "<plan_title>", "<text>", "<title_data>"
]

# Donut encodes JSON automatically if tokens are present. Add start and end variants.
json_tokens = []
for token in schema_tokens:
    json_tokens.append(token)
    json_tokens.append(token.replace("<", "</"))

new_tokens = task_start_tokens + json_tokens

processor.tokenizer.add_tokens(new_tokens)

# 3. Model Customization
model = VisionEncoderDecoderModel.from_pretrained(model_id)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# Configure tokens
model.config.pad_token_id = processor.tokenizer.pad_token_id
# We set the decoder_start_token_id to a default task or let the processor handle prompt prepending
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(["<s_general>"])[0] 
model.config.eos_token_id = processor.tokenizer.eos_token_id

print("Processor and Model configured.")


### Dataset Preparation
Transform the raw dataset into model inputs (pixel values, decoder input ids).


In [ ]:
from torch.utils.data import Dataset, DataLoader
import re

class DonutDataset(Dataset):
    def __init__(self, dataset, processor, max_length=512, split="train", task_prompt="<s_general>"):
        super().__init__()
        self.dataset = dataset
        self.processor = processor
        self.max_length = max_length
        self.split = split
        self.task_prompt = task_prompt

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image'].convert('RGB')
        
        # Determine actual task prompt based on the ground truth (can be customized)
        # Here we just use a default, but for single-model generalization, 
        # you'd extract the root node of gt_parse (e.g. if 'lot_geometry' is present, use <s_lot_geometry>)
        prompt = self.task_prompt
        
        gt = item['ground_truth']
        if isinstance(gt, str):
            gt = json.loads(gt)
            
        gt_parse = gt.get('gt_parse', gt)
        
        # If ground truth has a single top-level key acting as task:
        if isinstance(gt_parse, dict) and len(gt_parse) == 1:
            root_key = list(gt_parse.keys())[0]
            if f"<s_{root_key}>" in task_start_tokens:
                prompt = f"<s_{root_key}>"
                
        # Format the text
        sequence = prompt + self.processor.token2json(gt_parse) + processor.tokenizer.eos_token
        
        # Processor preprocessing
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()
        
        # Tokenize labels
        labels = self.processor.tokenizer(
            sequence, 
            add_special_tokens=False, 
            max_length=self.max_length, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )["input_ids"].squeeze()
        
        # Ignore padding
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {"pixel_values": pixel_values, "labels": labels}

train_ds = DonutDataset(train_dataset, processor)
val_ds = DonutDataset(val_dataset, processor, split="val")

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False, num_workers=2)


### PyTorch Lightning Trainer
Define the LightningModule and validation logic (Edit Distance).


In [ ]:
import pytorch_lightning as pl
import editdistance

class DonutModelPLModule(pl.LightningModule):
    def __init__(self, model, processor, learning_rate=3e-5):
        super().__init__()
        self.model = model
        self.processor = processor
        self.learning_rate = learning_rate

    def training_step(self, batch, batch_idx):
        pixel_values = batch["pixel_values"]
        labels = batch["labels"]
        outputs = self.model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        pixel_values = batch["pixel_values"]
        labels = batch["labels"].clone()
        labels[labels == -100] = self.processor.tokenizer.pad_token_id

        # Pass prompt for generation. Assuming batch size of N and we take the first label token as prompt
        prompt_ids = labels[:, :1] 
        
        outputs = self.model.generate(
            pixel_values,
            decoder_input_ids=prompt_ids,
            max_length=self.model.decoder.config.max_position_embeddings,
            pad_token_id=self.processor.tokenizer.pad_token_id,
            eos_token_id=self.processor.tokenizer.eos_token_id,
            use_cache=True,
            bad_words_ids=[[self.processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )
        
        preds = self.processor.tokenizer.batch_decode(outputs.sequences, skip_special_tokens=True)
        gts = self.processor.tokenizer.batch_decode(labels, skip_special_tokens=True)
        
        edit_dist = 0
        cer_dist = 0
        total_chars = 0
        
        for pred, gt in zip(preds, gts):
            gt_text = gt.strip()
            pred_text = pred.strip()
            # Calculate metrics
            edit_dist += editdistance.eval(pred_text, gt_text)
            cer_dist += editdistance.eval(pred_text, gt_text)
            total_chars += len(gt_text)
            
        self.log("val_edit_distance", edit_dist / max(1, len(preds)))
        self.log("val_cer", cer_dist / max(1, total_chars))
        return edit_dist

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        # Custom LR scheduler
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.8)
        return [optimizer], [scheduler]

# Initialize PL module
pl_module = DonutModelPLModule(model, processor)

# Initialize trainer
trainer = pl.Trainer(
    max_epochs=10,
    accelerator="gpu",
    devices=1,
    gradient_clip_val=1.0,
    precision=16, # Mixed precision for faster training
)

print("Starting training!")
# Uncomment to train
# trainer.fit(pl_module, train_loader, val_loader)


## Phase III: Export & Verification
Save the model to Google Drive for local deployment and run an inference test.


In [ ]:
# Save Export Logic
EXPORT_DIR = '/content/drive/MyDrive/SurveyPlan_AI/Models/Donut_Finetuned'

print(f"Saving model and processor to {EXPORT_DIR}...")
# Uncomment after training
# model.save_pretrained(EXPORT_DIR)
# processor.save_pretrained(EXPORT_DIR)
print("Saved!")


In [ ]:
# Inference Test
def run_inference(image, task_prompt="<s_general>"):
    # Prepare inputs
    pixel_values = processor(image, return_tensors="pt").pixel_values
    
    # Prepend task token
    decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids
    
    # Generate
    outputs = model.generate(
        pixel_values.to(model.device),
        decoder_input_ids=decoder_input_ids.to(model.device),
        max_length=model.decoder.config.max_position_embeddings,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        use_cache=True,
        bad_words_ids=[[processor.tokenizer.unk_token_id]],
        return_dict_in_generate=True,
    )
    
    sequence = processor.batch_decode(outputs.sequences)[0]
    sequence = sequence.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
    sequence = re.sub(r"<.*?>", "", sequence, count=1).strip()  # remove first task start token
    
    # Convert back to JSON
    extracted_json = processor.token2json(sequence)
    return extracted_json

# Example:
# sample_image = val_dataset[0]['image'].convert('RGB')
# result = run_inference(sample_image, task_prompt="<s_lot_geometry>")
# print(json.dumps(result, indent=2))
